# Notebook 03 — Model Training

Interactive walkthrough of the BiLSTM classifier training process.

**What we cover:**
1. Sequence dataset overview (shapes, class balance)
2. Model architecture summary
3. Training loop with live loss tracking
4. Learning rate and dropout effect exploration
5. Training curves visualization
6. Quick inference sanity check

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
import torch
import torch.nn.functional as F

from src.modeling.bilstm_model import RunningFormClassifier, build_model, count_parameters

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

CLASS_ORDER  = ['good_form', 'overstriding', 'forward_lean', 'arm_crossing']
CLASS_COLORS = ['#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
SEQ_DIR      = Path('../data/processed/sequences')
print('✅ Imports OK')

## 1. Sequence Dataset Overview

In [ ]:
if (SEQ_DIR / 'X_train.npy').exists():
    X_train = np.load(SEQ_DIR / 'X_train.npy')
    X_val   = np.load(SEQ_DIR / 'X_val.npy')
    X_test  = np.load(SEQ_DIR / 'X_test.npy')
    y_train = np.load(SEQ_DIR / 'y_train.npy')
    y_val   = np.load(SEQ_DIR / 'y_val.npy')
    y_test  = np.load(SEQ_DIR / 'y_test.npy')
    
    with open(SEQ_DIR / 'label_encoder.pkl', 'rb') as f:
        le = pickle.load(f)
    
    print('SEQUENCE SHAPES')
    print(f'  X_train : {X_train.shape}  (windows × frames × features)')
    print(f'  X_val   : {X_val.shape}')
    print(f'  X_test  : {X_test.shape}')
    print(f'  Classes : {le.classes_.tolist()}')
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, y, name in [(axes[0], y_train, 'Train'),
                         (axes[1], y_val,   'Val'),
                         (axes[2], y_test,  'Test')]:
        counts = np.bincount(y, minlength=4)
        ax.bar(CLASS_ORDER, counts, color=CLASS_COLORS, edgecolor='white', width=0.6)
        for i, c in enumerate(counts):
            ax.text(i, c + 0.3, str(c), ha='center', fontweight='bold', fontsize=9)
        ax.set_title(f'{name} ({sum(counts)} windows)')
        ax.tick_params(axis='x', rotation=20)
    
    plt.suptitle('Class Distribution per Split', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Run build_sequences.py first.')

## 2. Model Architecture

In [ ]:
if 'X_train' in dir():
    n_features = X_train.shape[2]
    model = build_model(input_size=n_features, hidden_size=128, num_layers=2, num_classes=4)
    print(f'Input size  : {n_features} features')
    print(f'Seq length  : {X_train.shape[1]} frames')
    print(f'Parameters  : {count_parameters(model):,}')
    print()
    print(model)
    
    # Smoke test
    x_dummy = torch.randn(4, X_train.shape[1], n_features)
    with torch.no_grad():
        logits, attn = model(x_dummy, return_attention=True)
    probs = F.softmax(logits, dim=1)
    print(f'\nSmoke test:')
    print(f'  Output shape  : {logits.shape}')
    print(f'  Probs sum     : {probs.sum(1).mean():.4f} (should be 1.0)')
    print(f'  Attention     : {attn.shape}')

## 3. Training Curves (from saved run)

If you've already run `train_classifier.py`, visualise the training history.

In [ ]:
curves_path = Path('../models/classifier/training_curves.png')
if curves_path.exists():
    from PIL import Image
    img = Image.open(curves_path)
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Training curves not yet generated.')
    print('Run: python src/modeling/train_classifier.py')

## 4. Effect of Hyperparameters

Compare model capacity (hidden size) effect on parameter count.

In [ ]:
if 'X_train' in dir():
    hidden_sizes  = [32, 64, 128, 256]
    params_1layer = [count_parameters(RunningFormClassifier(n_features, h, 1, 4)) for h in hidden_sizes]
    params_2layer = [count_parameters(RunningFormClassifier(n_features, h, 2, 4)) for h in hidden_sizes]
    
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hidden_sizes, [p/1000 for p in params_1layer], 'o-',
            color='#4C72B0', label='1 LSTM layer', linewidth=2)
    ax.plot(hidden_sizes, [p/1000 for p in params_2layer], 's-',
            color='#C44E52', label='2 LSTM layers', linewidth=2)
    ax.axvline(128, color='gray', linestyle='--', alpha=0.5, label='Default (128)')
    ax.set_xlabel('Hidden Size (per direction)')
    ax.set_ylabel('Parameters (thousands)')
    ax.set_title('Model Capacity vs Hidden Size', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print('Recommended: hidden=128, layers=2 → good balance of capacity and regularisation')

## 5. Inference Sanity Check on Test Set

In [ ]:
ckpt_path = Path('../models/classifier/best_model.pt')
if ckpt_path.exists() and 'X_test' in dir():
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    model = RunningFormClassifier(
        input_size=ckpt['input_size'],
        hidden_size=ckpt['hidden_size'],
        num_layers=ckpt['num_layers'],
        num_classes=ckpt['num_classes'],
        dropout=ckpt['dropout'],
    )
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    
    X_t = torch.FloatTensor(X_test[:20])
    with torch.no_grad():
        logits, attn = model(X_t, return_attention=True)
    probs = F.softmax(logits, dim=1).numpy()
    preds = probs.argmax(1)
    
    print('First 20 test samples:')
    print(f'{"True":25s} {"Predicted":25s} {"Confidence":10s}')
    print('-' * 62)
    for i in range(min(20, len(preds))):
        true_cls  = le.classes_[y_test[i]]
        pred_cls  = le.classes_[preds[i]]
        conf      = probs[i].max()
        ok_marker = '✅' if preds[i] == y_test[i] else '❌'
        print(f'{ok_marker} {true_cls:23s} → {pred_cls:23s}  {conf:.2%}')
else:
    print('Train the model first: python src/modeling/train_classifier.py')